# 눈치코치 · KoELECTRA 학습
Colab에서 런타임 유형을 T4 GPU로 바꾸고, 왼쪽 파일 패널에서 `NoonchiCoach-Colab.zip`과
`kocosa_preprocessed.parquet`를 /content에 업로드합니다. 전처리 파일은 기존 베이스라인에 사용한 파일입니다.
원본 Train/Validation/Test 파일은 전처리 Parquet를 다시 만들 때만 필요합니다.
학습 중간 체크포인트는 Colab 임시 디스크를 사용합니다. 마지막 셀은 필요할 때만 실행해 결과를 개인 Drive에 저장합니다.

In [ ]:
from pathlib import Path
import hashlib
import subprocess
import sys
import zipfile

root = Path('/content/noonchi')
archive_path = Path('/content/NoonchiCoach-Colab.zip')
data = Path('/content/kocosa_preprocessed.parquet')
for path in (archive_path, data):
    if not path.is_file():
        raise FileNotFoundError(f'Colab 파일 패널로 업로드하세요: {path.name}')
if hashlib.sha256(data.read_bytes()).hexdigest() != '3997f717a50b20e65bfd15090d6f9b7f091a6d14e13296f90608e969a913f433':
    raise ValueError('베이스라인에 사용한 전처리 Parquet와 파일 내용이 다릅니다.')
subprocess.run(['nvidia-smi'], check=True)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(root / 'requirements-colab.txt')], check=True)

In [ ]:
from datetime import datetime, timezone

result = Path('/content/noonchi-runs') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
subprocess.run([sys.executable, '-u', str(root / 'finetune.py'), str(data),
                '--output', str(result), '--batch-size', '8'], check=True)
# CUDA 메모리 부족 시 위 셀의 배치를 4 또는 2로 바꾸어 재실행합니다.

In [ ]:
import json
from IPython.display import Image, display

print(json.loads((result / 'metrics.json').read_text(encoding='utf-8'))['test'])
for name in ('confusion.png', 'validation.png'):
    display(Image(filename=str(result / name)))

In [ ]:
# 선택: 성공한 실험의 결과 한 벌만 개인 Google Drive에 저장합니다.
from google.colab import drive
import shutil

if not (result / 'metrics.json').is_file() or not (result / 'final/model.safetensors').is_file():
    raise RuntimeError('학습과 평가를 먼저 완료하세요.')
drive.mount('/content/drive')
destination = Path('/content/drive/MyDrive/NoonchiCoach') / result.name
required_bytes = sum(p.stat().st_size for p in result.rglob('*') if p.is_file())
print(f'저장 용량: {required_bytes / 1_000_000:.1f} MB')
if required_bytes > min(4_000_000_000, shutil.disk_usage(destination.parents[1]).free):
    raise RuntimeError('Drive 여유 공간을 확인하세요.')
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(result, destination)
print(destination)